In [39]:
# pip install pandas 
! pip install numpy

# Step 1. Data understanding：

In [40]:
import pandas as pd
import numpy as np

In [41]:
df = pd.read_csv("stock_data_350.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])

# keep recent 5 years
df = df[df["Date"] >= "2020-01-01"].copy()

In [42]:
df

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume
0,2020-01-02,A,85.900002,86.349998,85.199997,85.949997,82.210464,1410500.0
1,2020-01-03,A,84.669998,85.330002,84.500000,84.570000,80.890480,1118300.0
2,2020-01-06,A,84.000000,84.820000,83.599998,84.820000,81.129623,1993200.0
3,2020-01-07,A,83.959999,85.260002,83.940002,85.080002,81.378319,1684700.0
4,2020-01-08,A,85.959999,86.470001,85.199997,85.919998,82.181763,1847600.0
...,...,...,...,...,...,...,...,...
433658,2024-12-24,XOM,106.519997,107.190002,105.699997,106.400002,101.953667,7807000.0
433659,2024-12-26,XOM,106.519997,107.029999,105.940002,106.489998,102.039902,9652400.0
433660,2024-12-27,XOM,106.300003,107.989998,105.769997,106.480003,102.030327,11943900.0
433661,2024-12-30,XOM,106.300003,106.559998,105.510002,105.760002,101.340408,11080800.0


In [43]:
df.shape

(433663, 8)

In [44]:
df.columns

Index(['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close',
       'Volume'],
      dtype='str')

In [45]:
df.dtypes

Date         datetime64[us]
Ticker                  str
Open                float64
High                float64
Low                 float64
Close               float64
Adj Close           float64
Volume              float64
dtype: object

In [46]:
df.head()

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume
0,2020-01-02,A,85.900002,86.349998,85.199997,85.949997,82.210464,1410500.0
1,2020-01-03,A,84.669998,85.330002,84.500000,84.570000,80.890480,1118300.0
2,2020-01-06,A,84.000000,84.820000,83.599998,84.820000,81.129623,1993200.0
3,2020-01-07,A,83.959999,85.260002,83.940002,85.080002,81.378319,1684700.0
4,2020-01-08,A,85.959999,86.470001,85.199997,85.919998,82.181763,1847600.0


In [47]:
df["Date"].min(), df["Date"].max()

(Timestamp('2020-01-02 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [48]:
df["Ticker"].nunique()

346

In [49]:
df["Ticker"].unique()

<StringArray>
[   'A', 'AAPL', 'ABBV', 'ABNB',  'ABT', 'ACGL',  'ACN', 'ADBE',  'ADI',
  'ADM',
 ...
  'TXN',  'UNH',  'UNP',  'UPS',  'USB',    'V',   'VZ',  'WFC',  'WMT',
  'XOM']
Length: 346, dtype: str

In [50]:
df.isnull().sum()

Date         0
Ticker       0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

In [51]:
df.duplicated().sum()

np.int64(0)

In [52]:
df.groupby("Ticker").size().describe()

count     346.000000
mean     1253.361272
std        50.392599
min       513.000000
25%      1258.000000
50%      1258.000000
75%      1258.000000
max      1258.000000
dtype: float64

In [53]:
df.groupby("Ticker").size().sort_values().head(10)

Ticker
GEHC     513
CEG      742
ABNB    1020
OTIS    1205
CARR    1205
TSLA    1258
TMUS    1258
TGT     1258
T       1258
STZ     1258
dtype: int64

In [54]:
df[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].describe()

,Open,High,Low,Close,Adj Close,Volume
count,433663.000000,433663.000000,433663.000000,433663.000000,433663.000000,4.336630e+05
mean,148.554354,150.309311,146.766184,148.561315,140.953857,8.036504e+06
std,183.887562,185.956816,181.758246,183.910249,181.712072,3.010858e+07
min,4.040000,4.160000,3.800000,4.010000,3.009994,0.000000e+00
25%,54.970001,55.639999,54.230000,54.950001,49.535585,1.198500e+06
50%,99.209999,100.430000,97.989998,99.190002,92.491440,2.568000e+06
75%,181.929993,184.059998,179.679993,181.886864,171.260925,6.094250e+06
max,3369.000000,3416.709961,3345.439941,3370.270020,3370.270020,1.543911e+09


# Step 2. Data cleaning：

In [55]:
# 2. convert date
df["Date"] = pd.to_datetime(df["Date"])

In [56]:
# 3. sort by ticker and date
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)


In [57]:
# 4. basic checks
print("Original shape:", df.shape)
print("Number of tickers:", df["Ticker"].nunique())
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Ticker-Date rows:", df.duplicated(subset=["Ticker", "Date"]).sum())

Original shape: (433663, 8)
Number of tickers: 346
Missing values:
 Date         0
Ticker       0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64
Duplicate rows: 0
Duplicate Ticker-Date rows: 0


In [58]:
# 5. remove rows with invalid numeric values
price_cols = ["Open", "High", "Low", "Close", "Adj Close"]

for col in price_cols:
    df = df[df[col] > 0]

# remove zero or negative volume
df = df[df["Volume"] > 0]


In [59]:
# 6. remove stocks with too short history
obs_count = df.groupby("Ticker").size()
valid_tickers = obs_count[obs_count >= 1200].index

removed_tickers = obs_count[obs_count < 1200].index.tolist()
print("Removed tickers (history too short):", removed_tickers)

df_clean = df[df["Ticker"].isin(valid_tickers)].copy()

Removed tickers (history too short): ['ABNB', 'CEG', 'GEHC']


In [60]:
# 7. final check
print("Cleaned shape:", df_clean.shape)
print("Remaining tickers:", df_clean["Ticker"].nunique())

print("\nObservation count after cleaning:")
print(df_clean.groupby("Ticker").size().describe())

Cleaned shape: (431386, 8)
Remaining tickers: 343

Observation count after cleaning:
count     343.000000
mean     1257.685131
std         4.041450
min      1205.000000
25%      1258.000000
50%      1258.000000
75%      1258.000000
max      1258.000000
dtype: float64


In [61]:
# 每只股票记录数是否更一致
df_clean.groupby("Ticker").size().sort_values().head(10)

Ticker
OTIS    1205
CARR    1205
CNC     1257
CCI     1257
UNP     1258
UNH     1258
TXN     1258
TSLA    1258
TMUS    1258
TGT     1258
dtype: int64

# Step 3. Feature engineering：

In [62]:
# daily return
df_clean["daily_return"] = df_clean.groupby("Ticker")["Adj Close"].pct_change()

In [63]:
# 2. create daily-level features
df_clean["daily_return"] = df_clean.groupby("Ticker")["Adj Close"].pct_change()
df_clean["abs_return"] = df_clean["daily_return"].abs()
df_clean["log_volume"] = np.log1p(df_clean["Volume"])
df_clean["price_range"] = (df_clean["High"] - df_clean["Low"]) / df_clean["Open"]

In [64]:
# 3. create market proxy return
market_proxy = (
    df_clean.groupby("Date")["daily_return"]
      .mean()
      .reset_index()
      .rename(columns={"daily_return": "market_proxy_return"})
)

df_clean = df_clean.merge(market_proxy, on="Date", how="left")

In [65]:
# 4. remove rows where daily_return is NaN
# (usually the first day for each ticker)
df_model = df_clean.dropna(subset=["daily_return"]).copy()

In [66]:
# 5. stock-level summary features
feature_df = df_model.groupby("Ticker").agg(
    mean_return=("daily_return", "mean"),
    return_volatility=("daily_return", "std"),
    avg_abs_return=("abs_return", "mean"),
    positive_return_ratio=("daily_return", lambda x: (x > 0).mean()),
    avg_log_volume=("log_volume", "mean"),
    volume_volatility=("log_volume", "std"),
    price_range_mean=("price_range", "mean")
).reset_index()

In [67]:
# 6. trend return over full sample period
trend_df = (
    df_clean.groupby("Ticker")
      .agg(
          first_adj_close=("Adj Close", "first"),
          last_adj_close=("Adj Close", "last")
      )
      .reset_index()
)

trend_df["trend_return"] = (
    trend_df["last_adj_close"] / trend_df["first_adj_close"] - 1
)

trend_df = trend_df[["Ticker", "trend_return"]]


In [68]:
# 7. market correlation
corr_list = []

for ticker, group in df_model.groupby("Ticker"):
    corr = group["daily_return"].corr(group["market_proxy_return"])
    corr_list.append([ticker, corr])

corr_df = pd.DataFrame(corr_list, columns=["Ticker", "market_corr"])

In [69]:
# 8. merge all features
feature_df = feature_df.merge(trend_df, on="Ticker", how="left")
feature_df = feature_df.merge(corr_df, on="Ticker", how="left")

In [70]:
# 9. final check
print(feature_df.shape)
print(feature_df.head())
print(feature_df.isnull().sum())

(343, 10)
  Ticker  mean_return  return_volatility  avg_abs_return  \
0      A     0.000557           0.018689        0.013490   
1   AAPL     0.001182           0.019956        0.014064   
2   ABBV     0.000839           0.015613        0.010644   
3    ABT     0.000414           0.016469        0.011417   
4   ACGL     0.000870           0.021334        0.014212   

   positive_return_ratio  avg_log_volume  volume_volatility  price_range_mean  \
0               0.528242       14.288650           0.397404          0.022991   
1               0.533015       18.193167           0.484622          0.022455   
2               0.543357       15.627346           0.420658          0.020144   
3               0.511535       15.466207           0.382363          0.019713   
4               0.537788       14.372062           0.448544          0.024015   

   trend_return  market_corr  
0      0.617359     0.672563  
1      2.440023     0.669260  
2      1.459009     0.478774  
3      0.419376   

# Step 4. Exploratory analysis：

# Step 5. Preprocessing for clustering：

# Step 6. Clustering：

# Step 7. Cluster interpretation：